# Paper 26: Forced-exit completeness

The live rule is already “fail a 10-Q, sell at the next open.” This notebook does not add a screen. It asks whether that sell actually finishes.

Three numbers, scored against v2:

1. **Name-days** the book still holds a name after the close that knew the filing.
2. **Extra slippage** versus the first real open after that close, plus the commission on the exit.
3. **NAV contribution** of the names still held five sessions later, on the days after the next-open sell should have flattened them.

`screen.breach_complete` turns on the retry. Left off, the book is v2 and the spec hash stays `6ca86844209e`. Turned on, in `monterey/sim/engine.py`:

- a breached name stays in the exit set until its shares are zero, including through the next month-end snapshot
- the order is the whole residual, so `int(round(...))` cannot leave a fraction behind
- the fill is the next real open, or that session's close if the open did not print. A forward-filled halt price is not used. Each session still held writes a `breach_wait` event

A name that is flat, then bought again because the next snapshot passes, is the paper 10 rule. The case file labels that `rebought`. Completeness is not supposed to remove it.


In [1]:
import subprocess
import sys
from pathlib import Path

def _repo_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "monterey" / "__init__.py").exists():
            return candidate
    raise FileNotFoundError(f"Monterey repo root not found from {start}")

_ROOT = _repo_root()
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))
try:
    import monterey  # noqa: F401
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", f"{_ROOT}[research]"])

import pandas as pd

from monterey.diagnostics.compliance import breach_handling
from monterey.research import boot, compare, run_book
from monterey.spec import BookSpec

NB_DIR = _ROOT / "Research" / "papers" / "26-forced-exit-completeness"
rd = boot("paper-26-forced-exit", start="2019-12-01", end="2026-09-30")
print("snapshot", rd.manifest.get("hash"))
print("v2 hash", BookSpec.load("fcf-sma-v2-baseline").hash())


[monterey.data] 631 symbols (503 current members)
[monterey.data] 1,147,266 price rows
[monterey.data] Scanning filings for AAOIFI breaches…
[monterey.data] Froze snapshot ebbbefb35545 → /Users/raoabdul/Documents/Development/Monterey-Finance/data/snapshots/ebbbefb35545
        field                   value
       window 2019-12-01 → 2026-09-30
     universe    sp500, point-in-time
      symbols                     631
    snapshots                      85
   price rows                 1147266
    dividends                    9754
filing events                   61274
   halalquant                   0.4.0
snapshot hash            ebbbefb35545
snapshot ebbbefb35545
v2 hash 6ca86844209e


## Case file

One row per breach event. `name_days` counts sessions with shares still on the book, starting at the close that knew the filing, and stops at the first flat close. The next-open rule is allowed **one** of those sessions. `shares_5` is the position five sessions later, the same check as `breach_handling`.

Labels on the rows that are still held at that fifth session:

| Label | What happened |
|---|---|
| `rebought` | Flat after the breach, then shares came back inside the five sessions. That is the next snapshot passing. |
| `dust` | A residual under half a share is still there. `round` produced no order. |
| `no_print` | The sell was rejected because there was no price. |
| `still_working` | Shares never went to zero, and neither of those two blocks explains it. |


In [2]:
def _sessions(ledger) -> pd.DatetimeIndex:
    return pd.DatetimeIndex(pd.to_datetime(ledger.nav["date"]).dt.normalize().unique()).sort_values()


def _share_map(positions: pd.DataFrame) -> pd.Series:
    if positions.empty:
        return pd.Series(dtype=float)
    frame = positions.copy()
    frame["date"] = pd.to_datetime(frame["date"]).dt.normalize()
    frame["shares"] = frame["shares"].astype(float)
    return frame.groupby(["date", "symbol"])["shares"].sum()


def _shares_at(shares: pd.Series, day, symbol: str) -> float:
    key = (pd.Timestamp(day).normalize(), symbol)
    if shares.empty or key not in shares.index:
        return 0.0
    return float(shares.loc[key])


def _sells(ledger) -> pd.DataFrame:
    fills = ledger.fills
    if fills.empty:
        return fills
    out = fills[(fills["side"] == "sell") & (fills["status"].isin(["filled", "partial", "rejected"]))].copy()
    out["date"] = pd.to_datetime(out["date"]).dt.normalize()
    return out


def breach_cases(ledger, lag: int = 5) -> pd.DataFrame:
    """One row per breach: how long the name stayed, and why it was still there at `lag`."""
    events = ledger.events
    breaches = events[events["type"] == "breach"].copy() if not events.empty else events
    if breaches is None or breaches.empty:
        return pd.DataFrame()
    breaches["date"] = pd.to_datetime(breaches["date"]).dt.normalize()
    sessions = _sessions(ledger)
    locate = {day: i for i, day in enumerate(sessions)}
    shares = _share_map(ledger.positions)
    sells = _sells(ledger)
    rows = []
    for event in breaches.itertuples(index=False):
        origin = pd.Timestamp(event.date)
        i = locate.get(origin)
        if i is None:
            continue
        symbol = str(event.symbol)

        def at(step: int):
            j = i + step
            if j >= len(sessions):
                return None
            return _shares_at(shares, sessions[j], symbol)

        window = []
        for step in range(lag + 1):
            held = at(step)
            if held is None:
                break
            window.append(held)
        spell = 0
        flat_on = None
        for step in range(0, len(sessions) - i):
            held = at(step)
            if held is None or held <= 0:
                flat_on = None if held is None else sessions[i + step]
                break
            spell += 1
        seen_flat = False
        came_back = False
        for held in window[1:]:
            if held <= 0:
                seen_flat = True
            elif seen_flat:
                came_back = True
                break
        later = sells[(sells["symbol"] == symbol) & (sells["date"] > origin)] if not sells.empty else sells
        if flat_on is not None and not later.empty:
            later = later[later["date"] <= pd.Timestamp(flat_on)]
        filled = later[later["status"].isin(["filled", "partial"])] if not later.empty else later
        rejected = later[later["status"] == "rejected"] if not later.empty else later
        first = filled.iloc[0] if not filled.empty else (rejected.iloc[0] if not rejected.empty else None)
        shares_5 = window[lag] if len(window) > lag else None
        rows.append(
            {
                "date": origin,
                "symbol": symbol,
                "detail": event.detail,
                "name_days": spell,
                "shares_0": window[0] if window else None,
                "shares_1": window[1] if len(window) > 1 else None,
                "shares_5": shares_5,
                "flat_on": flat_on,
                "came_back": came_back,
                "sell_date": None if first is None else first["date"],
                "sell_status": None if first is None else first["status"],
                "sell_detail": None if first is None else first["detail"],
                "sell_shares": None if first is None else float(first["shares"]),
                "sell_price": None if first is None else first["price"],
                "label": _label(shares_5, came_back, first, window),
            }
        )
    return pd.DataFrame(rows)


def _label(shares_5, came_back: bool, first, window: list[float]) -> str:
    if shares_5 is None:
        return "unchecked"
    if shares_5 <= 0:
        return "cleared"
    if came_back:
        return "rebought"
    if any(0 < held < 0.5 for held in window):
        return "dust"
    if first is None or first["status"] == "rejected" or first["detail"] in {"no print", "no price", "no close"}:
        return "no_print"
    return "still_working"


## Replay v2

The committed `diagnostics.json` reports zero breach events because `events.parquet` is not in git. The count below is the replay. `breach_handling` is the diagnostic behind “still held five sessions later.” The case file is the same events, one row each, so the two counts can be checked against each other.


In [3]:
base = run_book("fcf-sma-v2-baseline", rd, paper="26-forced-exit", id="v2-baseline")
print(base.spec.hash(), breach_handling(base.events, base.positions, lag=5))

cases = breach_cases(base)
print(cases["label"].value_counts().to_string())
stuck = cases[cases["label"].isin(["rebought", "dust", "no_print", "still_working"])]
stuck.sort_values(["label", "date", "symbol"])[
    ["date", "symbol", "label", "name_days", "shares_0", "shares_1", "shares_5", "sell_status", "sell_detail", "flat_on", "came_back"]
]


6ca86844209e {'events': 122, 'names': 46, 'checked': 104, 'still_held': 13}
label
cleared     109
rebought     13


,date,symbol,label,name_days,shares_0,shares_1,shares_5,sell_status,sell_detail,flat_on,came_back
7,2020-02-25,GILD,rebought,1,173.0000,0.0000,171.0000,filled,sell,2020-02-26,True
53,2023-02-22,GILD,rebought,1,123.0000,0.0000,123.0000,filled,sell,2023-02-23,True
54,2023-02-27,CME,rebought,1,34.0000,0.0000,36.0000,filled,sell,2023-02-28,True
61,2023-07-27,EBAY,rebought,1,48.0000,0.0000,50.0000,filled,sell,2023-07-28,True
70,2024-02-28,CME,rebought,1,40.0000,0.0000,40.0000,filled,sell,2024-02-29,True
80,2024-10-29,NDAQ,rebought,1,45.0000,0.0000,45.0000,filled,sell,2024-10-30,True
91,2025-02-27,CME,rebought,1,28.0000,0.0000,28.0000,filled,sell,2025-02-28,True
96,2025-07-25,CME,rebought,1,29.0000,0.0000,27.0000,filled,sell,2025-07-28,True
102,2025-10-30,ICE,rebought,1,57.0000,0.0000,51.0000,filled,sell,2025-10-31,True
110,2026-02-26,CME,rebought,1,30.0000,0.0000,31.0000,filled,sell,2026-02-27,True


## Name-days, slippage, NAV

**Name-days** sum `name_days` over every breach. **Extra name-days** subtract the one session the next-open rule is allowed to hold.

**Slippage** is cash left on the table versus the first *real* open after the breach close. A lower sell price is a positive cost. Commission is the `cost` on those sells. Both are divided by starting capital and shown in basis points. A halt that v2 sold on a forward-filled open can be flat inside five sessions, so it will not show up in `still_held`. It shows up here, as a price difference against the first open that actually printed. Completeness waits those sessions out, so its name-days rise by the wait and its slippage against that real open goes to zero.

**NAV contribution** uses only the names still held at five sessions. It starts the session *after* the next-open close (the book should already be flat) and adds `prior weight × that day's price return` until the name is flat. The sum is a fraction of NAV.


In [4]:
def first_real_open(rd, day, symbol: str):
    """First open after `day` that actually printed. A missing day is not filled forward."""
    panel = rd.prices.open
    later = rd.sessions[rd.sessions > pd.Timestamp(day)]
    for session in later:
        if panel.empty or session not in panel.index or symbol not in panel.columns:
            continue
        value = panel.at[session, symbol]
        try:
            number = float(value)
        except (TypeError, ValueError):
            continue
        if pd.notna(number) and number > 0:
            return session, number
    return None, None


def exit_cost(ledger, cases: pd.DataFrame, rd) -> dict:
    """Slippage and commission on the sells that flatten each breach spell."""
    sells = _sells(ledger)
    filled = sells[sells["status"].isin(["filled", "partial"])] if not sells.empty else sells
    slip = 0.0
    commission = 0.0
    priced = 0
    unfilled = 0
    for case in cases.itertuples(index=False):
        if case.flat_on is None and int(case.name_days) == 0:
            continue
        window = filled[(filled["symbol"] == case.symbol) & (filled["date"] > pd.Timestamp(case.date))] if not filled.empty else filled
        if case.flat_on is not None and not window.empty:
            window = window[window["date"] <= pd.Timestamp(case.flat_on)]
        if window.empty:
            if case.shares_5 is not None and case.shares_5 > 0:
                unfilled += 1
            continue
        _, open_px = first_real_open(rd, case.date, case.symbol)
        commission += float(window["cost"].astype(float).sum())
        if open_px is None:
            continue
        priced += 1
        for trade in window.itertuples(index=False):
            slip += float(trade.shares) * (open_px - float(trade.price))
    capital = float(ledger.spec.accounting["capital"])
    return {
        "name_days": int(cases["name_days"].sum()) if not cases.empty else 0,
        "extra_name_days": int((cases["name_days"] - 1).clip(lower=0).sum()) if not cases.empty else 0,
        "slip_dollars": slip,
        "slip_bps": slip / capital * 1e4 if capital else None,
        "commission_bps": commission / capital * 1e4 if capital else None,
        "exits_priced": priced,
        "still_unfilled": unfilled,
    }


def nav_contribution(ledger, cases: pd.DataFrame) -> float:
    """Return contribution of names still held five sessions later, after the allowed next-open day."""
    stuck = cases[cases["label"].isin(["rebought", "dust", "no_print", "still_working"])]
    if stuck.empty or ledger.positions.empty:
        return 0.0
    positions = ledger.positions.copy()
    positions["date"] = pd.to_datetime(positions["date"]).dt.normalize()
    total = 0.0
    for case in stuck.itertuples(index=False):
        window = positions[(positions["symbol"] == case.symbol) & (positions["date"] >= pd.Timestamp(case.date))].sort_values("date")
        rows = []
        for rec in window.itertuples(index=False):
            rows.append(rec)
            if pd.Timestamp(rec.date) > pd.Timestamp(case.date) and float(rec.shares) <= 0:
                break
        # rows[0] is the breach close. The next close is the allowed exit day.
        # Pairs after that are days the book should already have been flat.
        for prev, cur in zip(rows[1:], rows[2:]):
            if float(prev.shares) <= 0 or not prev.price or not cur.price or float(prev.price) <= 0:
                continue
            total += float(prev.weight) * (float(cur.price) / float(prev.price) - 1.0)
    return float(total)


## Completeness book

Same v2 spec with `breach_complete` on. The case file is rebuilt from that ledger so name-days are counted on the path the retry actually traded.


In [5]:
complete = run_book(
    "fcf-sma-v2-baseline",
    rd,
    paper="26-forced-exit",
    id="breach-complete",
    screen={"breach_complete": True},
)
print(complete.spec.hash(), breach_handling(complete.events, complete.positions, lag=5))
complete_cases = breach_cases(complete)
print(complete_cases["label"].value_counts().to_string())


af9fe1e86411 {'events': 122, 'names': 46, 'checked': 104, 'still_held': 13}
label
cleared     109
rebought     13


## The three numbers

`compare` is underneath. This paper is not promoted on Calmar. A Calmar gap that comes from staying long a name that failed the screen is the leak being measured above.


In [6]:
base_cost = exit_cost(base, cases, rd)
complete_cost = exit_cost(complete, complete_cases, rd)
base_nav = nav_contribution(base, cases)
complete_nav = nav_contribution(complete, complete_cases)

summary = pd.DataFrame(
    [
        {
            "book": "v2-baseline",
            "name_days": base_cost["name_days"],
            "extra_name_days": base_cost["extra_name_days"],
            "slip_bps": base_cost["slip_bps"],
            "commission_bps": base_cost["commission_bps"],
            "still_unfilled": base_cost["still_unfilled"],
            "stuck_nav_bps": base_nav * 1e4,
            "still_held_5": int((cases["label"].isin(["rebought", "dust", "no_print", "still_working"])).sum()),
        },
        {
            "book": "breach-complete",
            "name_days": complete_cost["name_days"],
            "extra_name_days": complete_cost["extra_name_days"],
            "slip_bps": complete_cost["slip_bps"],
            "commission_bps": complete_cost["commission_bps"],
            "still_unfilled": complete_cost["still_unfilled"],
            "stuck_nav_bps": complete_nav * 1e4,
            "still_held_5": int((complete_cases["label"].isin(["rebought", "dust", "no_print", "still_working"])).sum()),
        },
    ]
).set_index("book")

day_cut = base_cost["extra_name_days"] - complete_cost["extra_name_days"]
slip_gap = (complete_cost["slip_bps"] or 0) - (base_cost["slip_bps"] or 0)
rebought = int((cases["label"] == "rebought").sum())
leaks = int(cases["label"].isin(["dust", "no_print", "still_working"]).sum())
# Positive slip_gap: the retry sold for less than v2. Positive day_cut: the retry held fewer extra sessions.
if leaks == 0 and rebought and day_cut == 0:
    decision = (
        f"The names still held at five sessions are rebuys after a flat close ({rebought}). "
        "That is the next snapshot passing, which paper 10 already allows. Completeness does not change it."
    )
elif day_cut > 0 and slip_gap <= 1:
    decision = (
        f"Retries cut extra breach-days by {day_cut:.0f} and the slippage gap is {slip_gap:.2f} bp. "
        "That is small enough to ship as a live ops rule."
    )
elif day_cut > 0:
    decision = (
        f"Retries cut extra breach-days by {day_cut:.0f}. The slippage gap is {slip_gap:.2f} bp against the book. "
        "Flatten anyway. A failed name is not a position we keep because the exit was inconvenient."
    )
elif day_cut < 0 and slip_gap <= 1:
    decision = (
        f"Extra breach-days rose by {-day_cut:.0f} because the book waited for a real print "
        f"instead of a forward-filled halt. Slippage against that real open moved {slip_gap:.2f} bp. "
        "The wait is the compliance cost of refusing a stale price. Ship the flag."
    )
elif day_cut < 0:
    decision = (
        f"Waiting for a real print added {-day_cut:.0f} extra breach-days and {slip_gap:.2f} bp of slippage. "
        "Disclose the cost. Flatten anyway. A failed name is not kept because the next print gaps down."
    )
else:
    decision = (
        f"Extra breach-days were unchanged and the slippage gap is {slip_gap:.2f} bp. "
        f"v2 labels: {cases['label'].value_counts().to_dict()}. Read the stuck table before turning the flag on in the live book."
    )

cases.to_csv(NB_DIR / "breach_cases.csv", index=False)
complete_cases.to_csv(NB_DIR / "breach_cases_complete.csv", index=False)
summary.to_csv(NB_DIR / "exit_completeness.csv")
print(decision)
summary


The names still held at five sessions are rebuys after a flat close (13). That is the next snapshot passing, which paper 10 already allows. Completeness does not change it.


,name_days,extra_name_days,slip_bps,commission_bps,still_unfilled,stuck_nav_bps,still_held_5
book,,,,,,,
v2-baseline,102,0,0.0000,6.8195,0,0.0000,13
breach-complete,102,0,0.0000,6.8107,0,0.0000,13


In [7]:
table = compare([base, complete], data=rd)
table


,total_return,cagr,volatility,max_drawdown,calmar,sharpe,sortino,turnover,costs_pct,dividends_pct,avg_exposure,switches,hash
book,,,,,,,,,,,,,
v2-baseline,1.0644,0.1137,0.1463,-0.2482,0.4582,0.6752,0.8589,4.1653,0.0561,0.0561,0.8032,36.0000,6ca86844209e
breach-complete,1.0634,0.1136,0.1462,-0.2481,0.4581,0.6749,0.8586,4.1503,0.0559,0.0561,0.8032,36.0000,af9fe1e86411
SPUS (total return),2.1167,0.1840,0.2116,-0.3080,0.5975,0.8123,1.0828,NaN,NaN,NaN,NaN,NaN,NaN
SPY (total return),1.6113,0.1533,0.2006,-0.3372,0.4547,0.7140,0.8854,NaN,NaN,NaN,NaN,NaN,NaN
